# 🎓 Simulador de Sistema Acadêmico de Notas

Notebook para o Google Colab. Basta executar as células em ordem (**Ambiente de execução → Executar tudo**).

**O que ele faz**
1. Simula um sistema acadêmico: **6 turmas × 30 alunos × 10 matérias × 8 semestres**, com nome, nota, turno, classe, matéria, professor, semestre e idade.
2. Calcula: média, média móvel, máximas/mínimas, 2º e 3º quartis, série histórica, densidade, dispersão, intervalo de confiança, p-valor (95%) e análise de sensibilidade.
3. **Valida matematicamente** o simulador (valores teóricos × empíricos, cobertura do IC, calibração do p-valor, normalidade, lei dos grandes números).

**Modelo gerador (verdade conhecida)**

$$\text{nota}_{ijst} = \text{clip}\Big(\underbrace{\mu + m_j + p_j}_{\text{matéria + professor}} + \underbrace{\tau_{turno(i)}}_{\text{turno}} + \underbrace{\beta\,(idade_i-22)}_{\text{idade}} + \underbrace{\gamma\, s}_{\text{tendência}} + \underbrace{a_i}_{\text{habilidade}} + \underbrace{\varepsilon_{ijst}}_{\text{ruído}},\;0,\;10\Big)$$

com $a_i \sim N(0,\sigma_a^2)$ e $\varepsilon \sim N(0,\sigma_\varepsilon^2)$. Como sabemos a "verdade", podemos comparar o que o simulador produz com o que a teoria prevê.

## 1. Ambiente e pacotes

In [ ]:
import importlib.util, subprocess, sys

# Instala apenas o que estiver faltando (no Colab tudo isso já vem instalado)
for pkg, mod in {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy",
                 "matplotlib": "matplotlib", "seaborn": "seaborn"}.items():
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import os, shutil
from dataclasses import dataclass, field, replace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

try:
    from IPython.display import display
except ImportError:          # fora do Jupyter
    display = print

sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold"})

os.makedirs("figs", exist_ok=True)
def salvar(nome):
    plt.savefig(f"figs/{nome}.png", dpi=200, bbox_inches="tight")

ALPHA = 0.05   # nível de significância (confiança de 95%)

print("Python", sys.version.split()[0])
import scipy, matplotlib
print(f"numpy {np.__version__} | pandas {pd.__version__} | scipy {scipy.__version__} | "
      f"matplotlib {matplotlib.__version__} | seaborn {sns.__version__}")

## 2. Configuração e gerador de dados

In [ ]:
@dataclass
class Config:
    n_turmas: int = 6
    n_alunos_por_turma: int = 30
    n_materias: int = 10
    n_semestres: int = 8
    mu_base: float = 6.5          # nota média "de base"
    sd_materia: float = 0.6       # variação entre dificuldades das matérias
    sd_prof: float = 0.4          # viés de cada professor
    sd_aluno: float = 1.0         # σ_a  (habilidade persistente do aluno)
    sd_ruido: float = 1.0         # σ_ε  (ruído de cada avaliação)
    efeito_turno: dict = field(default_factory=lambda: {"Manhã": 0.3, "Tarde": 0.0, "Noite": -0.4})
    escala_turno: float = 1.0     # multiplica o efeito do turno (0 = turno não importa)
    tendencia_semestre: float = 0.05   # ganho médio de nota por semestre
    efeito_idade: float = -0.03   # variação de nota por ano de idade (ref.: 22 anos)
    clip: bool = True             # limita notas em [0, 10]
    seed: int = 42

PRIMEIROS = ["Ana","Bruno","Carla","Diego","Elisa","Felipe","Gabriela","Hugo","Isabela","João",
             "Karen","Lucas","Marina","Nicolas","Olívia","Pedro","Quitéria","Rafael","Sofia","Tiago",
             "Ursula","Victor","Wesley","Yasmin","Zeca","Amanda","Caio","Daniela","Eduardo","Fernanda",
             "Gustavo","Helena","Igor","Júlia","Leonardo","Mariana","Natália","Otávio","Paula","Renato"]
SOBRENOMES = ["Silva","Santos","Oliveira","Souza","Lima","Pereira","Costa","Ferreira","Rodrigues","Almeida",
              "Nascimento","Carvalho","Araújo","Ribeiro","Gomes","Martins","Barbosa","Rocha","Dias","Moreira",
              "Cardoso","Teixeira","Cavalcanti","Nunes","Mendes","Freitas","Pinto","Castro","Campos","Monteiro",
              "Vieira","Lopes","Machado","Moura","Correia","Azevedo","Melo","Batista","Farias","Medeiros"]
MATERIAS = ["Cálculo I","Estatística","Programação","Banco de Dados","Física","Álgebra Linear",
            "Inglês Técnico","Redes de Computadores","Engenharia de Software","Ética e Sociedade"]
PROFESSORES = ["Prof. Almeida","Profa. Barros","Prof. Cavalcanti","Profa. Duarte","Prof. Esteves",
               "Profa. Fonseca","Prof. Guedes","Profa. Honório","Prof. Ibrahim","Profa. Jardim"]


def gerar_dados(cfg: Config, com_nomes: bool = True) -> pd.DataFrame:
    """Gera a base 'longa': uma linha por (semestre, aluno, matéria)."""
    rng = np.random.default_rng(cfg.seed)
    T, A, M, S = cfg.n_turmas, cfg.n_alunos_por_turma, cfg.n_materias, cfg.n_semestres
    N = T * A

    # --- estrutura: turmas (classes A, B, ...) e turnos (2 turmas por turno) ---
    turnos = list(cfg.efeito_turno)
    classes = [chr(65 + i) for i in range(T)]
    turno_da_classe = [turnos[i * len(turnos) // T] for i in range(T)]
    classe_aluno = np.repeat(classes, A)
    turno_aluno = np.repeat(turno_da_classe, A)
    ef_turno = np.array([cfg.efeito_turno[t] for t in turno_aluno]) * cfg.escala_turno

    # --- atributos dos alunos ---
    idade = rng.integers(18, 31, size=N)
    habilidade = rng.normal(0, cfg.sd_aluno, N)                 # a_i

    # --- matérias e professores (1 professor por matéria) ---
    ef_materia = rng.normal(0, cfg.sd_materia, M)
    ef_prof = rng.normal(0, cfg.sd_prof, M)
    nomes_mat = (MATERIAS + [f"Matéria {k+1}" for k in range(len(MATERIAS), M)])[:M]
    nomes_prof = (PROFESSORES + [f"Prof. {k+1}" for k in range(len(PROFESSORES), M)])[:M]

    # --- parte determinística: E[nota bruta]  (shape: S x N x M) ---
    mu = ((cfg.mu_base + ef_materia + ef_prof)[None, None, :]
          + ef_turno[None, :, None]
          + cfg.efeito_idade * (idade - 22)[None, :, None]
          + cfg.tendencia_semestre * np.arange(S)[:, None, None])

    ruido = rng.normal(0, cfg.sd_ruido, (S, N, M))               # ε
    bruta = mu + habilidade[None, :, None] + ruido
    nota = np.round(np.clip(bruta, 0, 10), 2) if cfg.clip else bruta

    # --- nomes (RNG separado para não alterar as notas) ---
    if com_nomes:
        rn = np.random.default_rng(cfg.seed + 1)
        total = len(PRIMEIROS) * len(SOBRENOMES)
        c = rn.choice(total, size=N, replace=N > total)
        nomes = np.array([f"{PRIMEIROS[k // len(SOBRENOMES)]} {SOBRENOMES[k % len(SOBRENOMES)]}" for k in c])
    else:
        nomes = np.array([f"Aluno {i+1:04d}" for i in range(N)])

    s_i, n_i, m_i = (a.ravel() for a in np.indices((S, N, M)))
    return pd.DataFrame({
        "semestre": s_i + 1,
        "aluno_id": n_i + 1,
        "nome": nomes[n_i],
        "idade": idade[n_i],
        "classe": classe_aluno[n_i],
        "turno": turno_aluno[n_i],
        "materia": np.array(nomes_mat)[m_i],
        "professor": np.array(nomes_prof)[m_i],
        "nota": nota.ravel(),
        "mu_teorico": mu.ravel(),    # E[nota bruta] segundo o modelo (usado só na validação)
    })


def por_aluno(df):
    """Nível do aluno (unidade independente): média de todas as notas."""
    return df.groupby("aluno_id").agg(media=("nota", "mean"), idade=("idade", "first"),
                                      classe=("classe", "first"), turno=("turno", "first"),
                                      nome=("nome", "first"))


def ic_t(x, conf=0.95):
    """IC t de Student para a média: retorna (média, limite inferior, limite superior)."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    m, se = x.mean(), x.std(ddof=1) / np.sqrt(n)
    lo, hi = stats.t.interval(conf, n - 1, loc=m, scale=se)
    return m, lo, hi


cfg = Config()
df = gerar_dados(cfg)
print(f"Base gerada: {len(df):,} linhas  |  {df.aluno_id.nunique()} alunos  |  "
      f"{df.classe.nunique()} turmas  |  {df.materia.nunique()} matérias  |  {df.semestre.nunique()} semestres")
display(df.drop(columns="mu_teorico").head(10))

## 3. Estatísticas descritivas: média, máx/mín, 2º e 3º quartis

In [ ]:
AGG = dict(media="mean", desvio="std", minimo="min", Q2_mediana="median",
           Q3=lambda x: x.quantile(0.75), maximo="max")

tab_mat = df.groupby("materia")["nota"].agg(**AGG)
tab_geral = df.assign(materia="GERAL").groupby("materia")["nota"].agg(**AGG)
tab_materia = pd.concat([tab_mat, tab_geral]).round(2)
print("Por matéria:")
display(tab_materia)

tab_turma = df.groupby(["turno", "classe"])["nota"].agg(**AGG).round(2)
print("Por turno e turma:")
display(tab_turma)

fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
ordem = tab_mat.sort_values("media").index
sns.boxplot(data=df, y="materia", x="nota", order=ordem, ax=ax[0], color="#8ecae6", fliersize=2)
ax[0].set_title("Distribuição das notas por matéria (caixa = Q1–Q2–Q3)")
ax[0].set_xlabel("Nota"); ax[0].set_ylabel("")

med_turma = df.groupby(["classe", "turno"])["nota"].mean().reset_index()
sns.barplot(data=med_turma, x="classe", y="nota", hue="turno", dodge=False, ax=ax[1])
ax[1].set_ylim(5, 8); ax[1].set_title("Média por turma (cor = turno)")
ax[1].set_xlabel("Turma"); ax[1].set_ylabel("Nota média")
plt.tight_layout(); salvar("01_descritivas"); plt.show()

## 4. Série histórica e média móvel

In [ ]:
JANELA = 3
serie = df.groupby("semestre")["nota"].agg(media="mean", minimo="min", maximo="max")
serie["media_movel"] = serie["media"].rolling(JANELA, min_periods=1).mean()
display(serie.round(3))

pivot_sm = df.pivot_table(index="semestre", columns="materia", values="nota", aggfunc="mean")

fig, ax = plt.subplots(figsize=(10, 5.5))
for col in pivot_sm.columns:
    ax.plot(pivot_sm.index, pivot_sm[col], color="gray", alpha=0.25, lw=1)
ax.plot([], [], color="gray", alpha=0.4, label="Médias por matéria")
ax.plot(serie.index, serie["media"], "o-", color="C0", lw=2.5, label="Média geral (série histórica)")
ax.plot(serie.index, serie["media_movel"], "s--", color="C3", lw=2.5, label=f"Média móvel ({JANELA} semestres)")
ax.set_xlabel("Semestre"); ax.set_ylabel("Nota média"); ax.set_title("Série histórica das notas")
ax.legend(); plt.tight_layout(); salvar("02_serie_historica"); plt.show()

## 5. Gráfico de densidade

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(df["nota"], stat="density", bins=40, alpha=0.3, ax=ax[0], color="C0")
sns.kdeplot(df["nota"], ax=ax[0], color="C0", lw=2.5, clip=(0, 10), label="Densidade (KDE)")
xs = np.linspace(0, 10, 300)
ax[0].plot(xs, stats.norm.pdf(xs, df["nota"].mean(), df["nota"].std()), "r--", label="Normal ajustada")
ax[0].axvline(df["nota"].mean(), color="k", ls=":", label="Média")
ax[0].set_title("Densidade das notas (todas as observações)")
ax[0].set_xlabel("Nota"); ax[0].legend()

sns.kdeplot(data=df, x="nota", hue="turno", fill=True, alpha=0.3, common_norm=False,
            clip=(0, 10), ax=ax[1], hue_order=["Manhã", "Tarde", "Noite"])
ax[1].set_title("Densidade por turno"); ax[1].set_xlabel("Nota")
plt.tight_layout(); salvar("03_densidade"); plt.show()

## 6. Gráfico de dispersão

In [ ]:
aluno = por_aluno(df)
piv_aluno_mat = df.pivot_table(index="aluno_id", columns="materia", values="nota", aggfunc="mean")
m1, m2 = "Cálculo I", "Estatística"

r1, p1 = stats.pearsonr(aluno["idade"], aluno["media"])
r2, p2 = stats.pearsonr(piv_aluno_mat[m1], piv_aluno_mat[m2])

fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
sns.regplot(data=aluno, x="idade", y="media", x_jitter=0.25, ax=ax[0],
            scatter_kws=dict(alpha=0.5, s=22), line_kws=dict(color="C3"))
ax[0].set_title(f"Idade × nota média do aluno  (r = {r1:.2f}, p = {p1:.3f})")
ax[0].set_xlabel("Idade (anos)"); ax[0].set_ylabel("Nota média do aluno")

sns.scatterplot(x=piv_aluno_mat[m1], y=piv_aluno_mat[m2], hue=aluno["turno"], ax=ax[1], s=30, alpha=0.7)
sns.regplot(x=piv_aluno_mat[m1], y=piv_aluno_mat[m2], scatter=False, ax=ax[1], color="k")
ax[1].set_title(f"{m1} × {m2}, por aluno  (r = {r2:.2f}, p = {p2:.1e})")
ax[1].set_xlabel(f"Nota média em {m1}"); ax[1].set_ylabel(f"Nota média em {m2}")
plt.tight_layout(); salvar("04_dispersao"); plt.show()

## 7. Intervalo de confiança (95%)

As linhas da base **não são independentes** (o mesmo aluno aparece em várias matérias e semestres). Por isso o **aluno** é a unidade amostral: o IC é calculado sobre a média de cada aluno (n = 180).

In [ ]:
linhas = []
for mat in piv_aluno_mat.columns:
    m, lo, hi = ic_t(piv_aluno_mat[mat])
    linhas.append(dict(materia=mat, media=m, ic_inf=lo, ic_sup=hi, largura=hi - lo))
ic_mat = pd.DataFrame(linhas).sort_values("media").reset_index(drop=True)
display(ic_mat.round(3))

# IC global: t de Student e bootstrap por aluno (reamostra alunos inteiros)
medias = aluno["media"].to_numpy()
m_g, lo_t, hi_t = ic_t(medias)
rng_b = np.random.default_rng(0)
boot = rng_b.choice(medias, size=(5000, len(medias)), replace=True).mean(axis=1)
lo_b, hi_b = np.percentile(boot, [2.5, 97.5])
print(f"Média geral = {m_g:.3f}")
print(f"IC95% (t de Student) = [{lo_t:.3f}, {hi_t:.3f}]")
print(f"IC95% (bootstrap)    = [{lo_b:.3f}, {hi_b:.3f}]")

fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
ax[0].errorbar(ic_mat["media"], ic_mat["materia"], xerr=[ic_mat["media"] - ic_mat["ic_inf"], ic_mat["ic_sup"] - ic_mat["media"]],
               fmt="o", capsize=4, color="C0")
ax[0].axvline(m_g, color="r", ls="--", label="Média geral")
ax[0].set_title("Média por matéria com IC 95%"); ax[0].set_xlabel("Nota"); ax[0].legend()

sns.histplot(boot, bins=40, ax=ax[1], color="C0", alpha=0.5)
for v, c, l in [(lo_t, "r", "IC t"), (hi_t, "r", None), (lo_b, "g", "IC bootstrap"), (hi_b, "g", None)]:
    ax[1].axvline(v, color=c, ls="--", label=l)
ax[1].set_title("Distribuição bootstrap da média geral"); ax[1].set_xlabel("Média"); ax[1].legend()
plt.tight_layout(); salvar("05_intervalo_confianca"); plt.show()

## 8. Testes de hipótese e p-valor (α = 0,05 → 95% de confiança)

In [ ]:
testes = []
def registrar(nome, h0, estat, p):
    testes.append(dict(teste=nome, H0=h0, estatistica=estat, p_valor=p,
                       decisao_95="Rejeita H0" if p < ALPHA else "Não rejeita H0"))

manha = aluno.loc[aluno.turno == "Manhã", "media"]
noite = aluno.loc[aluno.turno == "Noite", "media"]
t = stats.ttest_ind(manha, noite, equal_var=False)
registrar("t de Welch: Manhã × Noite", "médias iguais", t.statistic, t.pvalue)

f = stats.f_oneway(*[g["media"].to_numpy() for _, g in aluno.groupby("classe")])
registrar("ANOVA entre as 6 turmas", "todas as turmas têm a mesma média", f.statistic, f.pvalue)

r, p = stats.pearsonr(aluno["idade"], aluno["media"])
registrar("Correlação de Pearson: idade × nota", "correlação = 0", r, p)

pv = df.pivot_table(index="aluno_id", columns="semestre", values="nota", aggfunc="mean")
t = stats.ttest_rel(pv.iloc[:, -1], pv.iloc[:, 0])
registrar("t pareado: último × primeiro semestre", "sem evolução", t.statistic, t.pvalue)

t = stats.ttest_1samp(aluno["media"], 6.0)
registrar("t de 1 amostra: média geral vs 6,0", "média = 6,0", t.statistic, t.pvalue)

tab_testes = pd.DataFrame(testes)
display(tab_testes.round(4))

fig, ax = plt.subplots(figsize=(10, 4.5))
cores = ["C3" if p < ALPHA else "gray" for p in tab_testes["p_valor"]]
ax.barh(tab_testes["teste"], tab_testes["p_valor"].clip(lower=1e-16), color=cores)
ax.set_xscale("log"); ax.axvline(ALPHA, color="k", ls="--", label="α = 0,05")
ax.set_xlabel("p-valor (escala log)"); ax.set_title("p-valores (vermelho = significativo a 95%)")
ax.legend(); plt.tight_layout(); salvar("06_pvalores"); plt.show()

## 9. Análise de sensibilidade

Varia **um parâmetro por vez** (mantendo os outros no padrão) e repete a simulação 40× por valor. Mede:
- **média global** e sua variação entre réplicas;
- **largura do IC 95%** (precisão);
- **poder** do teste Manhã × Noite (proporção de réplicas com p < 0,05).

In [ ]:
def simular_uma(c):
    d = gerar_dados(c, com_nomes=False)
    al = d.groupby("aluno_id").agg(media=("nota", "mean"), turno=("turno", "first"))
    m, lo, hi = ic_t(al["media"])
    p = stats.ttest_ind(al.loc[al.turno == "Manhã", "media"], al.loc[al.turno == "Noite", "media"],
                        equal_var=False).pvalue
    return dict(media=m, largura=hi - lo, rejeita=p < ALPHA)

def experimento(c, R=40, seed0=1000):
    r = pd.DataFrame([simular_uma(replace(c, seed=seed0 + k)) for k in range(R)])
    return dict(media_global=r["media"].mean(), sd_media=r["media"].std(),
                largura_ic=r["largura"].mean(), poder=r["rejeita"].mean())

GRADE = {
    "mu_base":            [5.0, 6.0, 6.5, 7.5, 8.5],
    "sd_aluno":           [0.5, 1.0, 1.5, 2.0, 3.0],
    "sd_ruido":           [0.5, 1.0, 1.5, 2.0, 3.0],
    "escala_turno":       [0.0, 0.5, 1.0, 2.0, 3.0],
    "n_alunos_por_turma": [10, 20, 30, 60, 100],
}
linhas = []
for param, valores in GRADE.items():
    for v in valores:
        linhas.append(dict(parametro=param, valor=v, **experimento(replace(cfg, **{param: v}))))
sens = pd.DataFrame(linhas)
display(sens.round(3))

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, (param, g) in zip(axes.ravel(), sens.groupby("parametro", sort=False)):
    ax.plot(g["valor"], g["poder"], "o-", color="C0")
    ax.set_ylim(0, 1.05); ax.set_ylabel("poder (P[p < 0,05])", color="C0"); ax.set_xlabel(param)
    ax2 = ax.twinx(); ax2.grid(False)
    ax2.plot(g["valor"], g["largura_ic"], "s--", color="C3")
    ax2.set_ylabel("largura do IC 95%", color="C3")
    ax.set_title(f"Sensibilidade a {param}")

amp = sens.groupby("parametro", sort=False)["media_global"].agg(lambda s: s.max() - s.min())
ax = axes.ravel()[-1]
ax.barh(amp.index, amp.values, color="C2")
ax.set_title("Amplitude da média global\n(máx − mín ao variar o parâmetro)"); ax.set_xlabel("Δ nota média")
plt.tight_layout(); salvar("07_sensibilidade"); plt.show()

## 10. Validação matemática do simulador

Como o modelo gerador é conhecido, comparamos o que o código produz com o que a **teoria** prevê:

| Bloco | O que é verificado |
|---|---|
| A | Estrutura e invariantes (6×30 alunos, 10 matérias, notas em [0,10], sem nulos) |
| B | Fórmulas calculadas "à mão" = funções do numpy/pandas (média, desvio, quartis, média móvel, IC) |
| C | Simulação grande **sem truncar** notas: média, variância, correlação entre matérias (ICC), normalidade e tendência batem com a teoria |
| D | Monte Carlo (1.000 réplicas): **cobertura do IC95% ≈ 95%** e **p-valor uniforme sob H0** (erro tipo I ≈ 5%) |
| E | Reprodutibilidade (mesma semente → mesma base) |

In [ ]:
checks = []
def check(nome, obtido, esperado, tol, relativo=False):
    erro = abs(obtido - esperado) / (abs(esperado) if relativo else 1)
    checks.append(dict(bloco=nome[0], verificacao=nome[1:].strip(), obtido=obtido, esperado=esperado,
                       tolerancia=tol, resultado="PASS" if erro <= tol else "FAIL"))
def check_bool(nome, cond):
    checks.append(dict(bloco=nome[0], verificacao=nome[1:].strip(), obtido=bool(cond), esperado=True,
                       tolerancia="-", resultado="PASS" if cond else "FAIL"))

# ---------- A. Estrutura ----------
T, A, M, S = cfg.n_turmas, cfg.n_alunos_por_turma, cfg.n_materias, cfg.n_semestres
check_bool("A  nº de linhas = turmas × alunos × matérias × semestres", len(df) == T * A * M * S)
check_bool("A  180 alunos únicos (6 × 30)", df.aluno_id.nunique() == T * A)
check_bool("A  10 matérias e 6 turmas", df.materia.nunique() == 10 and df.classe.nunique() == 6)
check_bool("A  30 alunos em cada turma", (df.groupby("classe")["aluno_id"].nunique() == A).all())
check_bool("A  notas dentro de [0, 10]", df["nota"].between(0, 10).all())
check_bool("A  sem valores nulos", not df.drop(columns="mu_teorico").isna().any().any())
check_bool("A  cada aluno pertence a uma única turma", (df.groupby("aluno_id")["classe"].nunique() == 1).all())

# ---------- B. Fórmulas manuais ----------
x = df["nota"].to_numpy(); n = len(x)
media_man = x.sum() / n
check("B  média manual × numpy", media_man, x.mean(), 1e-12)
check("B  desvio-padrão amostral manual × numpy", np.sqrt(((x - media_man) ** 2).sum() / (n - 1)), x.std(ddof=1), 1e-10)

def quantil_manual(v, q):
    s = np.sort(v); pos = (len(s) - 1) * q; i = int(np.floor(pos)); fr = pos - i
    return s[i] + (s[min(i + 1, len(s) - 1)] - s[i]) * fr
check("B  Q2 (mediana) manual × numpy", quantil_manual(x, 0.5), np.percentile(x, 50), 1e-12)
check("B  Q3 manual × numpy", quantil_manual(x, 0.75), np.percentile(x, 75), 1e-12)

mm_manual = np.convolve(serie["media"].to_numpy(), np.ones(JANELA) / JANELA, mode="valid")
mm_pandas = serie["media"].rolling(JANELA).mean().dropna().to_numpy()
check("B  média móvel manual (convolução) × pandas", float(np.abs(mm_manual - mm_pandas).max()), 0.0, 1e-12)

se_man = medias.std(ddof=1) / np.sqrt(len(medias))
check("B  IC95% manual (m − t·EP) × função ic_t", medias.mean() - stats.t.ppf(0.975, len(medias) - 1) * se_man, lo_t, 1e-12)

# ---------- C. Simulação grande SEM truncar (valores teóricos conhecidos) ----------
cfg_v = replace(cfg, n_alunos_por_turma=500, clip=False, seed=2024)   # 3.000 alunos
dv = gerar_dados(cfg_v, com_nomes=False)
res = dv["nota"] - dv["mu_teorico"]                                    # deve ser N(0, σa² + σε²)

res_aluno = res.groupby(dv["aluno_id"]).mean()
z = res_aluno.mean() / (res_aluno.std(ddof=1) / np.sqrt(len(res_aluno)))
check("C  E[nota − μ_teórico] = 0 (z-score, |z| ≤ 3)", z, 0.0, 3.0)

var_teo = cfg_v.sd_aluno ** 2 + cfg_v.sd_ruido ** 2
check("C  Var(nota − μ) = σa² + σε²  (erro relativo)", res.var(), var_teo, 0.05, relativo=True)

mat0, mat1 = dv["materia"].unique()[:2]
msk0 = ((dv.semestre == 1) & (dv.materia == mat0)).to_numpy()
msk1 = ((dv.semestre == 1) & (dv.materia == mat1)).to_numpy()
x0, x1 = res[msk0].to_numpy(), res[msk1].to_numpy()
icc_teo = cfg_v.sd_aluno ** 2 / var_teo
check("C  correlação entre matérias = σa²/(σa²+σε²)", np.corrcoef(x0, x1)[0, 1], icc_teo, 0.05)

ks = stats.kstest(x0, stats.norm(loc=0, scale=np.sqrt(var_teo)).cdf)
check_bool(f"C  resíduos ~ Normal(0, σ) (KS, p = {ks.pvalue:.3f} > 0,05)", ks.pvalue > ALPHA)

ms = dv.groupby("semestre")["nota"].mean()
slope = np.polyfit(ms.index, ms.values, 1)[0]
check("C  tendência por semestre ≈ γ configurado", slope, cfg_v.tendencia_semestre, 0.02)

# ---------- D. Monte Carlo: cobertura do IC e calibração do p-valor ----------
cfg_mc = replace(cfg, escala_turno=0.0, n_semestres=2, clip=False)   # H0 verdadeira: turno não afeta a nota
R_MC = 1000
cobre, pvals = [], []
for k in range(R_MC):
    d = gerar_dados(replace(cfg_mc, seed=10_000 + k), com_nomes=False)
    al = d.groupby("aluno_id").agg(media=("nota", "mean"), turno=("turno", "first"))
    _, lo, hi = ic_t(al["media"])
    verdade = d["mu_teorico"].mean()
    cobre.append(lo <= verdade <= hi)
    pvals.append(stats.ttest_ind(al.loc[al.turno == "Manhã", "media"], al.loc[al.turno == "Noite", "media"],
                                 equal_var=False).pvalue)
pvals = np.array(pvals)
check(f"D  cobertura empírica do IC95% ({R_MC} réplicas)", float(np.mean(cobre)), 0.95, 0.025)
check("D  taxa de erro tipo I sob H0 (P[p<0,05])", float((pvals < ALPHA).mean()), ALPHA, 0.02)
ks_u = stats.kstest(pvals, "uniform")
check_bool(f"D  p-valores sob H0 ~ Uniforme(0,1) (KS, p = {ks_u.pvalue:.3f})", ks_u.pvalue > 0.01)

# ---------- E. Reprodutibilidade ----------
check_bool("E  mesma semente → base idêntica", gerar_dados(cfg).equals(gerar_dados(cfg)))
check_bool("E  sementes diferentes → bases diferentes", not gerar_dados(cfg).equals(gerar_dados(replace(cfg, seed=7))))

tab_val = pd.DataFrame(checks)
display(tab_val)
n_ok = (tab_val["resultado"] == "PASS").sum()
print(f"\n✅ {n_ok}/{len(tab_val)} verificações aprovadas")

In [ ]:
# Gráficos de apoio à validação
fig, ax = plt.subplots(1, 3, figsize=(17, 5))

sd_t = np.sqrt(var_teo)
stats.probplot(x0 / sd_t, dist="norm", plot=ax[0])
ax[0].set_title("QQ-plot dos resíduos (simulação sem truncar)")

nn = np.arange(1, len(x0) + 1)
ax[1].plot(nn, np.cumsum(x0) / nn, lw=1.5, label="média acumulada")
ax[1].plot(nn, 1.96 * sd_t / np.sqrt(nn), "r--", lw=1, label="±1,96·σ/√n")
ax[1].plot(nn, -1.96 * sd_t / np.sqrt(nn), "r--", lw=1)
ax[1].axhline(0, color="k", lw=0.8); ax[1].set_xscale("log")
ax[1].set_title("Lei dos grandes números"); ax[1].set_xlabel("nº de alunos (n)"); ax[1].legend()

sns.histplot(pvals, bins=20, stat="density", ax=ax[2], color="C0")
ax[2].axhline(1, color="r", ls="--", label="Uniforme(0,1)")
ax[2].set_title(f"p-valores sob H0 ({R_MC} réplicas)"); ax[2].set_xlabel("p-valor"); ax[2].legend()
plt.tight_layout(); salvar("08_validacao"); plt.show()

## 11. Exportar resultados

In [ ]:
df.drop(columns="mu_teorico").to_csv("notas_simuladas.csv", index=False, encoding="utf-8-sig")
with pd.ExcelWriter("resultados_simulador.xlsx") as xw:
    df.drop(columns="mu_teorico").to_excel(xw, sheet_name="dados", index=False)
    tab_materia.to_excel(xw, sheet_name="descritivas_materia")
    serie.to_excel(xw, sheet_name="serie_historica")
    ic_mat.to_excel(xw, sheet_name="ic95_materia", index=False)
    tab_testes.to_excel(xw, sheet_name="testes_p_valor", index=False)
    sens.to_excel(xw, sheet_name="sensibilidade", index=False)
    tab_val.to_excel(xw, sheet_name="validacao", index=False)
shutil.make_archive("figs", "zip", "figs")
print("Arquivos gerados: notas_simuladas.csv, resultados_simulador.xlsx, figs.zip")

try:                       # download automático no Colab
    from google.colab import files
    for arq in ["notas_simuladas.csv", "resultados_simulador.xlsx", "figs.zip"]:
        files.download(arq)
except ImportError:
    pass

## 12. Roteiro para a apresentação

**Como desenvolvi o simulador?** Modelo estatístico de efeitos aditivos (matéria, professor, turno, idade, semestre, habilidade do aluno e ruído gaussiano), gerado de forma vetorizada com NumPy a partir de uma semente (reprodutível). A saída é uma tabela longa com nome, nota, turno, classe, matéria, professor, semestre e idade.

**Que pacotes usei?** `numpy` (geração e vetorização), `pandas` (tabelas e agregações), `scipy.stats` (t, ANOVA, Pearson, KS, IC), `matplotlib` e `seaborn` (gráficos), `dataclasses` (parâmetros).

**Requisitos para funcionar:** Python ≥ 3.9 e os pacotes acima (já vêm no Google Colab; `openpyxl` só para exportar Excel). Não precisa de GPU nem de dados externos; roda em poucos minutos.

**Como validei matematicamente?** Seção 10: fórmulas manuais × bibliotecas; média, variância, correlação entre matérias e tendência × valores teóricos; normalidade (KS/QQ); lei dos grandes números; Monte Carlo com 1.000 réplicas mostrando cobertura do IC ≈ 95% e p-valor uniforme sob H0.

**Por que é importante?** Permite testar métodos de análise e dashboards acadêmicos sem expor dados reais de alunos (LGPD), com "verdade conhecida" para checar se as estatísticas estão corretas, e estudar cenários (turno, professor, tamanho de turma) por meio da análise de sensibilidade.